# NB00 — Ingest & consolidate

**Input:** `data/processed/*_panel.parquet` (60 per-company panels, already ingested)

**Output:** `panel_long.parquet`, `macro_q.parquet`, a missing-value ledger, and a W&B run `nb00-ingest-panel`

**Spec:** `docs/specs/nb00-ingest-and-consolidate.md` · **Plan:** `docs/plans/nb00-ingest-and-consolidate.md`

_Status: planned — boundaries only. Code cells are added by the plan's last task; this notebook never runs ingestion._

## E1 — is the 2026-Q2 row reported or projected?

**Answered 2026-09-23: reported.** 2026-Q2 (date 2026-06-30) is the last reported quarter and is the forecast origin. 2026-Q3 ends 2026-09-30, has not been reported, and is horizon 1. Panel evidence agrees: all 60 companies have 2026-Q2 revenue and none repeats the prior quarter. `Settings.LAST_REPORTED_QUARTER` holds this as data; bump it when a newer quarter is ingested.

**Known limitation — fiscal calendars.** Ingestion snaps each fiscal period end to the nearest calendar quarter end within 46 days, so companies with off-calendar fiscal years (AAPL, MSFT, PG, NKE, COST, DE, WMT, HD, M, BBY, ORCL, CSCO, ADBE, …) line up with macro data only within that margin. Not corrected here.

## 1. Load panels

Validated read of the 60 per-company files. **Exit:** 60 frames × 81 rows.

## 2. Consolidate and flag

`panel_long` (no macro, no `is_public`) + `macro_q` (81 rows); flags `covid`, `structural_break`, `outlier_flag`, `is_projected`. **Exit:** row count = sum of per-company rows (4,860); macro table exactly 81 rows.

## 3. Missing-value ledger

Every NaN cell with a reason. **Exit:** each `unexplained` row is fixed by re-ingestion or waived here in writing (expected today: COST EPS ×9, COST EBITDA 2026-06-30).

## 4. Save and track

Write both parquet files and log them as W&B dataset artifacts `panel_long` and `macro_q`.